In [210]:
#!pip install netCDF4

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import netCDF4 as nc
from netCDF4 import Dataset
from google.colab import drive, files

from sklearn.utils import resample

import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model, load_model

tf.test.gpu_device_name()
print(tf.__version__)

2.5.0


In [2]:
drive.mount('/content/gdrive')

Mounted at /content/gdrive


### Data importation

In [3]:
# Model Data
# model_data = np.load('/content/gdrive/My Drive/Colab Notebooks/model_data_new.npy') 
model_data = np.load('/content/gdrive/MyDrive/Colab Notebooks/ERA30Min/model_data_with_prec.npy') 

# Obs Data
obs_data = np.load('/content/gdrive/MyDrive/Colab Notebooks/ERA30Min/obs_data_new.npy') 

# Precomputed SZA
sza = np.load('/content/gdrive/MyDrive/Colab Notebooks/ERA30Min/sza_new.npy', allow_pickle=True)

test_ceil = np.load('/content/gdrive/MyDrive/Colab Notebooks/testing raw data/test_ceil.npy')
test_dew = np.load('/content/gdrive/MyDrive/Colab Notebooks/testing raw data/test_dew.npy')
test_prec = np.load('/content/gdrive/MyDrive/Colab Notebooks/testing raw data/test_prec.npy')
test_temp = np.load('/content/gdrive/MyDrive/Colab Notebooks/testing raw data/test_temp.npy')

In [23]:
ceil_fn = '/content/gdrive/MyDrive/Colab Notebooks/raw_vienna_data/ViennaCeiling.nc'

dew_fn = '/content/gdrive/MyDrive/Colab Notebooks/raw_vienna_data/ViennaDewpoint.nc'

prec_fn = '/content/gdrive/MyDrive/Colab Notebooks/raw_vienna_data/ViennaPrecipitation.nc'

temp_fn = '/content/gdrive/MyDrive/Colab Notebooks/raw_vienna_data/ViennaTemperature.nc'

uwind_fn = '/content/gdrive/MyDrive/Colab Notebooks/raw_vienna_data/ViennaUWind.nc'

vwind_fn = '/content/gdrive/MyDrive/Colab Notebooks/raw_vienna_data/ViennaVWind.nc'

In [24]:
print(len(sza))

197109


In [27]:
k = 103558

In [26]:
feat = nc.Dataset(vwind_fn)

print(feat)

lat = feat['latitude'][:]
long = feat['longitude'][:]

print(lat)
print(long)

feat_data = feat['v10']
print(feat_data.shape)

<class 'netCDF4._netCDF4.Dataset'>
root group (NETCDF3_64BIT_OFFSET data model, file format NETCDF3):
    Conventions: CF-1.6
    history: 2020-10-28 18:45:01 GMT by grib_to_netcdf-2.16.0: /opt/ecmwf/eccodes/bin/grib_to_netcdf -S param -o /cache/data0/adaptor.mars.internal-1603900733.5484767-21822-14-04a90d78-bed7-4440-b4c9-d49dccbda251.nc /cache/tmp/04a90d78-bed7-4440-b4c9-d49dccbda251-adaptor.mars.internal-1603900733.5489612-21822-3-tmp.grib
    dimensions(sizes): longitude(9), latitude(9), expver(2), time(103558)
    variables(dimensions): float32 longitude(longitude), float32 latitude(latitude), int32 expver(expver), int32 time(time), int16 v10(time, expver, latitude, longitude)
    groups: 
[49.   48.75 48.5  48.25 48.   47.75 47.5  47.25 47.  ]
[15.   15.25 15.5  15.75 16.   16.25 16.5  16.75 17.  ]
(103558, 2, 9, 9)


In [28]:
print(feat_data[0])

[[[0.9019156170137297 1.0289197398471208 1.171329688231308
   1.3704781648635487 1.753369292577234 2.3384149116647786
   2.9700537592474134 3.502494120356631 3.8188771719120016]
  [1.4869612361012743 1.0462043246114285 0.9703024523855557
   1.0112594032401108 1.1010640936459704 1.8138653392523107
   2.499612452184083 3.0639917199229982 3.4145681891642825]
  [-0.3733861371180151 -0.3940524884666438 -0.5620135621545902
   -0.3098840757013195 0.1665692608452487 1.2524920862550135
   2.2388406733486583 2.9967321400792795 3.21354095331853]
  [-0.8344336481137873 -0.4887419528276337 -0.3970585032082625
   -0.30499930174618906 -0.39781000689366725 0.3931476219947603
   1.3159941476717092 2.4605342605430396 2.8050987003010857]
  [0.6520406416166731 0.815116941349489 0.7437240912360441
   0.04144389722536923 -0.5680255916378276 0.054219459877248743
   0.8677221993278166 2.1520419976844174 2.4947276782289523]
  [1.3460542950878966 1.1168456710394685 0.4517649094563254
   -0.25352129929596834 -0.

In [29]:
def double_list(temp_list) :

  new_list = []

  length = len(temp_list)

  for i in range(length - 1) :

    new_list.append(temp_list[i])

    new_list.append((temp_list[i] + temp_list[i+1])/2)
    
  new_list.append(temp_list[length - 1])

  return np.array(new_list)

In [30]:
feat_list = []

for i in range(k) :

  temp = feat_data[i,0,4,5] + feat_data[i,0,4,6] + feat_data[i,0,4,7]

  temp = temp/3

  feat_list.append(temp)

In [31]:
feat_list = feat_list[:98555]

for i in range(98555) :

  if isinstance(feat_list[i], float) == False :

    feat_list[i] = np.nan

In [32]:
for i in range(1000) :

  print(feat_list[i])

1.0246612189631608
1.2989600641358696
0.9451270789244989
0.7504876244046863
0.3243850347802318
0.0450761650381585
-0.4170986014857207
-0.8472092107656669
-0.9155960461374928
-0.991998920820302
-1.0806763556980545
-1.9917493236369967
-2.717451382509451
-2.819029630653317
-2.463443136842669
-1.9660729477190035
-1.8561029084214518
-2.3216594415296523
-2.806755070458374
-3.623389075264795
-4.127147045714399
-4.065523743511216
-3.9983894142817307
-3.5806786158109625
-3.080928665016849
-2.47809745870806
-1.828673023904181
-1.47909855957677
-1.224589311453051
-1.1712325497893188
-0.9569287488347503
-0.8922994318899478
-0.7292231321571317
-0.9578055031343892
-1.2308518421647567
-1.1309018520059342
-1.2059269699321684
-1.3299250780239407
-1.6602109477592986
-1.3199050288852117
-1.312389992031165
-1.3854111001296527
-1.5631417217278603
-1.7031719084415993
-1.8974356111187094
-1.7989886283306962
-1.566147736469479
-1.4505414195313922
-1.2942286529672185
-1.482605576775325
-1.6319043089423886
-1.7

In [33]:
feat_series = pd.Series(feat_list)
temp = feat_series.interpolate()

temp[0] = temp[1] + (temp[1] - temp[2])

for i in range(1000) :

  print(temp[i])

print(temp[98554])

1.6527930493472405
1.2989600641358696
0.9451270789244989
0.7504876244046863
0.3243850347802318
0.0450761650381585
-0.4170986014857207
-0.8472092107656669
-0.9155960461374928
-0.991998920820302
-1.0806763556980545
-1.9917493236369967
-2.717451382509451
-2.819029630653317
-2.463443136842669
-1.9660729477190035
-1.8561029084214518
-2.3216594415296523
-2.806755070458374
-3.623389075264795
-4.127147045714399
-4.065523743511216
-3.9983894142817307
-3.5806786158109625
-3.080928665016849
-2.47809745870806
-1.828673023904181
-1.47909855957677
-1.224589311453051
-1.1712325497893188
-0.9569287488347503
-0.8922994318899478
-0.7292231321571317
-0.9578055031343892
-1.2308518421647567
-1.1309018520059342
-1.2059269699321684
-1.3299250780239407
-1.6602109477592986
-1.3199050288852117
-1.312389992031165
-1.3854111001296527
-1.5631417217278603
-1.7031719084415993
-1.8974356111187094
-1.7989886283306962
-1.566147736469479
-1.4505414195313922
-1.2942286529672185
-1.482605576775325
-1.6319043089423886
-1.7

In [34]:
feat_arr = double_list(temp)

print(feat_arr.shape)

np.save('test_uwind.npy', feat_arr)

(197109,)


In [ ]:
# Extract the individual features

# obs_data_vis = obs_data[:,0]
# obs_data_ceil = obs_data[:,1]
# obs_data_T = obs_data[:,2]
# obs_data_Td = obs_data[:,3]
# obs_data_ws = obs_data[:,4]
# obs_data_wd = obs_data[:,5]

# model_data_ceil = model_data[:,:,0]
# model_data_T = model_data[:,:,1]
# model_data_Td = model_data[:,:,2]
# model_data_ws = model_data[:,:,3]
# model_data_wd = model_data[:,:,4]
# model_data_prec = model_data[:,:,5]

# sza = np.repeat(np.expand_dims(sza, -1), 4, axis=1)

In [ ]:
model_data.shape

In [ ]:
# For simplicity, average the model data spatially
model_data = model_data.mean(axis=1)

In [ ]:
# Remove the model ceiling data
# model_data = model_data[:,1:]

# Use just the SZA as model data
# model_data = np.expand_dims(sza, axis=1)

# Select only a subset of features
model_data = np.column_stack((sza, test_temp, test_temp - test_dew, test_ceil, test_prec))

# # Select only a subset of features
# obs_data = np.column_stack((obs_data[:,0], obs_data[:,2], obs_data[:,2]-obs_data[:,3]))

In [ ]:
bins = [-1, 150, 350, 600, 800, 1500, 3000, 5000, 10000]
obs_data[:,0] = pd.cut(obs_data[:,0], bins, labels=[0,1,2,3,4,5,6,7]).to_numpy()

# Bin the data (second possibility)
# bins = [-1, 800, 1500, 3000, 5000, 8000, 10000]
# obs_data[:,0] = pd.cut(obs_data[:,0], bins, labels=[0,1,2,3,4,5]).to_numpy()

In [ ]:
num_classes = len(bins)-1

In [ ]:
# # Sanity check (what would happen if we'd use the actual visibility as model data -> perfect predictions!)
# obs_data = np.expand_dims(obs_data[:,0], axis=1)
# model_data = np.expand_dims(obs_data[:,0], axis=1)

In [ ]:
# Use the past_history nr of data to predict future_target nr of data
past_history = 6
future_target = 6

In [ ]:
# Indices to select
begin_index = [0] + [48*30*(10+12*i) for i in range(11)]
end_index = [48*30*(1+12*i) for i in range(11)] + [len(obs_data)]

indices = [[i,j] for i, j in zip(begin_index, end_index)]

In [ ]:
obs_data_winter = []
model_data_winter = []

for i in range(len(indices)):
  obs_data_winter.append(obs_data[indices[i][0]:indices[i][1],:])
  model_data_winter.append(model_data[indices[i][0]:indices[i][1],:])
obs_data_winter = np.concatenate(obs_data_winter, axis=0)
model_data_winter = np.concatenate(model_data_winter, axis=0)

In [ ]:
# Use only winter data
obs_data = obs_data_winter
model_data = model_data_winter

In [ ]:
# Split into training data
TRAIN_SPLIT = np.int(0.8*len(obs_data))
print(TRAIN_SPLIT)

### Data preparation

In [ ]:
def multivariate_data_model(dataset, target, start_index, end_index, history_size,
                      target_size, step, single_step=False):
  data = []
  labels = []

  start_index = start_index + history_size
  if end_index is None:
    end_index = len(dataset) - target_size

  for i in range(start_index, end_index):
    indices = range(i-history_size, i+target_size, step)
    data.append(dataset[indices])

    if single_step:
      labels.append(target[i+target_size])
    else:
      labels.append(target[i:i+target_size])

  return np.array(data), np.array(labels)


def multivariate_data_obs(dataset, target, start_index, end_index, history_size,
                      target_size, step, single_step=False):
  data = []
  labels = []

  start_index = start_index + history_size
  if end_index is None:
    end_index = len(dataset) - target_size

  for i in range(start_index, end_index):
    indices = range(i-history_size, i, step)
    data.append(dataset[indices])

    if single_step:
      labels.append(target[i+target_size])
    else:
      labels.append(target[i:i+target_size])

  return np.array(data), np.array(labels)

In [ ]:
STEP = 1

# Split into training and testing data
x_obs_train, y_obs_train = multivariate_data_obs(obs_data, obs_data, 0,
                                                 TRAIN_SPLIT, past_history,
                                                 future_target, STEP)
x_obs_test, y_obs_test = multivariate_data_obs(obs_data, obs_data,
                                             TRAIN_SPLIT, None, past_history,
                                             future_target, STEP)

x_model_train, _ = multivariate_data_model(model_data, model_data, 0,
                                                 TRAIN_SPLIT, past_history,
                                                 future_target, STEP)
x_model_test, _ = multivariate_data_model(model_data, model_data,
                                             TRAIN_SPLIT, None, past_history,
                                             future_target, STEP)

In [ ]:
# Normalize the data (all features except for visibility)
x_obs_min = x_obs_train.min(axis=(0,1))
x_obs_max = x_obs_train.max(axis=(0,1))
x_model_min = x_model_train.min(axis=(0,1))
x_model_max = x_model_train.max(axis=(0,1))

x_obs_train[:,:,1:] = (x_obs_train[:,:,1:]-x_obs_min[1:])/(x_obs_max[1:]-x_obs_min[1:])
x_obs_test[:,:,1:] = (x_obs_test[:,:,1:]-x_obs_min[1:])/(x_obs_max[1:]-x_obs_min[1:])

y_obs_train[:,:,1:] = (y_obs_train[:,:,1:]-x_obs_min[1:])/(x_obs_max[1:]-x_obs_min[1:])
y_obs_test[:,:,1:] = (y_obs_test[:,:,1:]-x_obs_min[1:])/(x_obs_max[1:]-x_obs_min[1:])

x_model_train = (x_model_train-x_model_min)/(x_model_max-x_model_min)
x_model_test = (x_model_test-x_model_min)/(x_model_max-x_model_min)

In [ ]:
# Implements over- and undersampling of the data
def resample_data(x_obs, y_obs, x_model, no_classes, samples_per_class):
  
  class_counts = np.zeros(no_classes).astype(np.int32)
  class_indices = []

  a = np.arange(0,len(y_obs),1)

  x_res = np.zeros((sum(samples_per_class), x_obs.shape[1], x_obs.shape[2]))
  y_res = np.zeros((sum(samples_per_class), y_obs.shape[1], y_obs.shape[2]))
  x_res_model = np.zeros((sum(samples_per_class), x_model.shape[1], x_model.shape[2]))

  k = 0
  for i in range(no_classes):

    # Indices of class i in dataset
    curr_class_index = a[np.any(y_obs[:,:,0]==i, axis=1)]

    # Resample class i to have precisely samples_per_class occurrences
    resampled_class_index = resample(curr_class_index, replace=True, 
                                     n_samples=samples_per_class[i])
    samples = len(resampled_class_index)

    x_res[k:k+samples,] = x_obs[resampled_class_index,]
    y_res[k:k+samples,] = y_obs[resampled_class_index,]
    x_res_model[k:k+samples,] = x_model[resampled_class_index,]
    k += samples

  return x_res, y_res, x_model

In [ ]:
# class_samples = [5000,100,1000,10000,1500,1000,8000,10000]
# x_obs_res, y_obs_res, x_model_res = resample_data(x_obs_train, y_obs_train, x_model_train, 8, class_samples)

In [ ]:
x_obs_res, y_obs_res, x_model_res = x_obs_train, y_obs_train, x_model_train

In [ ]:
plt.hist(y_obs_res[:,:,0], bins=num_classes)

In [ ]:
# Total number of model features
nr_features_model = x_model_res.shape[2]
nr_features_obs = x_obs_res.shape[2]

In [ ]:
# Data shape
print ('Single window of past history : {}'.format(x_model_res[0].shape))
print ('Target window to predict : {}'.format(y_obs_res[0].shape))

In [ ]:
from sklearn.utils import compute_class_weight

classes = [i for i in range(num_classes)]
weights = compute_class_weight('balanced', classes, obs_data[:,0])

class_weights = dict(zip(classes, weights))

In [ ]:
print(class_weights)

In [ ]:
def multi_layer_cross_entropy(y_true, y_pred):

  t = np.ones(future_target)
  fact = 1.0

  loss = t[0]*tf.keras.losses.sparse_categorical_crossentropy(y_true[:,0], y_pred[:,0,:]) + \
  fact*tf.cast(tf.keras.losses.mse(y_true[:,0], tf.math.argmax(y_pred[:,0,:], axis=1)), tf.float32)

  for i in range(1, future_target):
    y_true_step_i = y_true[:,i]
    y_pred_step_i = y_pred[:,i,:]
    loss += t[i]*tf.keras.losses.sparse_categorical_crossentropy(y_true_step_i, y_pred_step_i) + \
    fact*tf.cast(tf.keras.losses.mse(y_true_step_i, tf.math.argmax(y_pred_step_i, axis=1)), tf.float32)
  return loss

In [ ]:
def slice(x, length):
  return x[:,-length:,:]

def slice_and_scale_visibility(x):
  return x[:,-1, 0]/(num_classes-1)

In [ ]:
input_shape_obs = (past_history, nr_features_obs,)
input_shape_model = (past_history+future_target, nr_features_model,)

in_obs = tf.keras.layers.Input(shape=input_shape_obs)
in_model = tf.keras.layers.Input(shape=input_shape_model)

b_obs = tf.keras.layers.LSTM(16, return_sequences=True,)(in_obs)
b_obs = tf.keras.layers.Lambda(slice, arguments={'length': 1})(b_obs)
b_obs = tf.keras.layers.Flatten()(b_obs)
b_obs = tf.keras.layers.RepeatVector(future_target)(b_obs)
b_obs = tf.keras.layers.LSTM(num_classes, return_sequences=True)(b_obs)

b_model = tf.keras.layers.LSTM(16, return_sequences=True,)(in_model)
b_model = tf.keras.layers.Lambda(slice, arguments={'length': future_target})(b_model)
b_model = tf.keras.layers.LSTM(num_classes, return_sequences=True)(b_model)

b = tf.keras.layers.Add()([b_obs, b_model])

out = tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(num_classes, activation='softmax'))(b)

model = tf.keras.models.Model([in_obs, in_model], out)
model.compile(optimizer='adam', loss=multi_layer_cross_entropy, metrics=['acc'])
model.summary()

In [ ]:
multi_step_history = model.fit([x_obs_res, x_model_res], y_obs_res[:,:,0], 
                               batch_size=64, epochs=20,)
                               #  class_weight=class_weights)

In [ ]:
def plot_train_history(history, title):
  loss = history.history['loss']
  val_loss = history.history['val_loss']

  epochs = range(len(loss))

  plt.figure()

  plt.plot(epochs, loss, 'b', label='Training loss')
  plt.plot(epochs, val_loss, 'r', label='Validation loss')
  plt.title(title)
  plt.legend()

  plt.show()

In [ ]:
# plot_train_history(multi_step_history, 'Multi-Step Training and validation loss')

In [ ]:
# Plot some results
ix = np.random.randint(0, len(x_obs_test))

vis = np.expand_dims(x_obs_test[ix,:,0], axis=0)
true_vis = np.expand_dims(y_obs_test[ix,:,0], axis=0)

data_obs = np.expand_dims(x_obs_test[ix,:,:], axis=0)
data_model = np.expand_dims(x_model_test[ix,:,:], axis=0)

vis_prob = model([data_obs, data_model])
vis_mean = np.argmax(vis_prob, axis=2)

# This is the input observations
plt.plot(np.arange(-past_history+1, 1), vis[0,:past_history],'k--')

# Persitence prediction
plt.plot(np.arange(1, future_target+1), x_obs_test[ix, past_history-1,0]*np.ones(future_target),'g.')

# ML prediction
plt.plot(np.arange(1, future_target+1), vis_mean[0,],'b')

# True observed future
plt.plot(np.arange(1, future_target+1), true_vis[0,],'r--')

plt.legend(['Observed past','Persistence','ML model','Observed']) 
plt.grid()
plt.xlabel('time')
plt.ylabel('class')

# for i in range(runs):
#  plt.plot(np.arange(0, future_target), np.round(vis_prob[i,0,]),'r-.')

In [ ]:
def multi_layer_cross_entropy(y_true, y_pred):

  t = np.ones(future_target)

  loss = t[0]*tf.keras.losses.sparse_categorical_crossentropy(y_true[:,0], y_pred[:,0,:]) + \
  tf.cast(tf.keras.losses.mse(y_true[:,0], tf.math.argmax(y_pred[:,0,:], axis=1)), tf.float32)

  for i in range(1, future_target):
    y_true_step_i = y_true[:,i]
    y_pred_step_i = y_pred[:,i,:]
    loss += t[i]*tf.keras.losses.sparse_categorical_crossentropy(y_true_step_i, y_pred_step_i) + \
    tf.cast(tf.keras.losses.mse(y_true_step_i, tf.math.argmax(y_pred_step_i, axis=1)), tf.float32)
  return loss

In [ ]:
res = model([data_obs, data_model], training=True)[0,].numpy()
print(res.sum(axis=1))

In [ ]:
# Proper verification
from sklearn.metrics import *

In [ ]:
# Now verify the model
# 1) Persistence error
y_pers = np.repeat(np.expand_dims(x_obs_test[:, past_history-1,0], axis=1), future_target, axis=1)
err_pers = np.mean(np.square(y_pers - y_obs_test[:,:,0]), axis=0)

# 3) ML model error
vis_ml = model([x_obs_test, x_model_test])
y_ml = np.argmax(vis_ml, axis=2)
  
err_ml = np.mean(np.square(y_ml - y_obs_test[:,:,0]), axis=0)

print('Persistence error: {}'.format(err_pers))
print('ML model error: {}'.format(err_ml))

In [ ]:
plt.plot(np.arange(1,future_target+1,1), err_ml)
plt.plot(np.arange(1,future_target+1,1), err_pers)
plt.xlabel('Prediction step [30 minutes]')
plt.ylabel('MSE')
plt.legend(['ML error','Pers error'])

In [ ]:
nr_classes = num_classes

precision_pers = np.zeros((nr_classes,future_target))
recall_pers = np.zeros((nr_classes,future_target))
f1_pers = np.zeros((nr_classes,future_target))
balanced_acc_pers = np.zeros(future_target)

precision_ml = np.zeros((nr_classes,future_target))
recall_ml = np.zeros((nr_classes,future_target))
f1_ml = np.zeros((nr_classes,future_target))
balanced_acc_ml = np.zeros(future_target)

for i in range(future_target):
  precision_pers[:,i] = precision_score(y_obs_test[:,i,0], y_pers[:,i], 
                                        labels=[k for k in range(num_classes)], average=None)
  precision_ml[:,i] = precision_score(y_obs_test[:,i,0], y_ml[:,i], 
                                        labels=[k for k in range(num_classes)], average=None)

  recall_pers[:,i] = recall_score(y_obs_test[:,i,0], y_pers[:,i], 
                                        labels=[k for k in range(num_classes)], average=None)
  recall_ml[:,i] = recall_score(y_obs_test[:,i,0], y_ml[:,i], 
                                        labels=[k for k in range(num_classes)], average=None)

  f1_pers[:,i] = f1_score(y_obs_test[:,i,0], y_pers[:,i], 
                                        labels=[k for k in range(num_classes)], average=None)
  f1_ml[:,i] = f1_score(y_obs_test[:,i,0], y_ml[:,i], 
                                        labels=[k for k in range(num_classes)], average=None)
  
  balanced_acc_pers[i] = accuracy_score(y_obs_test[:,i,0], y_pers[:,i])
  balanced_acc_ml[i] = accuracy_score(y_obs_test[:,i,0], y_ml[:,i])

In [ ]:
def categorical_skill_scores(y_true, y_pred):

  # Compute the confusion matrix
  C = confusion_matrix(y_true, y_pred, labels=[i for i in range(num_classes)])
  C = C.transpose()
  nr_fcst = np.sum(C, axis=1)
  nr_obs = np.sum(C, axis=0)
  N = np.sum(nr_obs)

  # Heidke skill score
  HSS = (1/N*np.trace(C)-1/N**2*np.sum(nr_fcst*nr_obs))/(1-1/N**2*np.sum(nr_fcst*nr_obs))

  # Peirce skill score
  PSS = (1/N*np.trace(C)-1/N**2*np.sum(nr_fcst*nr_obs))/(1-1/N**2*np.sum(nr_obs**2))

  return HSS, PSS

In [ ]:
np.set_printoptions(precision=2)
for i in range(future_target):
  HSS, PSS = categorical_skill_scores(y_obs_test[:,i,0],y_ml[:,i])
  print('Prediction step {}:'.format(i+1))
  print('F1-score ML: {}'.format(f1_ml[:,i]))
  print('F1-score PE: {}'.format(f1_pers[:,i]))
  print('Accuracy ML: {}'.format(np.round(100*balanced_acc_ml[i])/100.0))
  print('Accuracy PE: {}'.format(np.round(100*balanced_acc_pers[i])/100.0))
  print('Heidke skill score ML: {}\nPeirce skill score ML: {}'.format(HSS, PSS))
  HSS, PSS = categorical_skill_scores(y_obs_test[:,i,0],y_pers[:,i])
  print('Heidke skill score PE: {}\nPeirce skill score PE: {}\n'.format(HSS, PSS))

In [ ]:
for i in range(future_target):
  print('Classification summary ML model at forecast hour {}:\n'.format(i+1))
  print(classification_report(y_obs_test[:,i,0], y_ml[:,i], labels=[0,1,2,3,4,5,6,7]))
  print('Confusion matrix:\n')
  print(confusion_matrix(y_obs_test[:,i,0], y_ml[:,i], labels=[0,1,2,3,4,5,6,7]))
  print('\n-----------------------------------------------------------')
  print('-----------------------------------------------------------\n')
  print('Classification summary Persistence at forecast hour {}:\n'.format(i+1))
  print(classification_report(y_obs_test[:,i,0], y_pers[:,i], labels=[k for k in range(num_classes)]))
  print('Confusion matrix:\n')
  print(confusion_matrix(y_obs_test[:,i,0], y_pers[:,i], labels=[k for k in range(num_classes)]))  
  print('\n')